# 第 15 章：Capstone——售后 Agent 工程验收

你已经分别学习了工具、状态、恢复、评估、服务、安全、RAG 和 Skills。现在把它们放到同一个业务边界中：查订单、读政策、获得人工审批、执行退款，服务重启后还能找回历史，并且重试不会多退一次。

| 读者契约 | 内容 |
|---|---|
| 你已会 | 前 14 章的主要概念；至少独立完成一次状态、恢复或安全练习 |
| 读完后你能 | 运行并解释真实 HTTP → ADK Runner → 工具 → 数据库链路；区分机制验收与模型质量评估 |
| 本章不碰 | 真实支付、分布式事务、公网身份系统与生产运维认证 |
| 主线 | 单机工程，默认离线模型，SQLite，无外部服务 |

**先猜：** 第一个进程退款成功后退出，第二个进程收到同样请求，应产生几条退款记录？只检查回答“成功”足够吗？

In [ ]:
from pathlib import Path
import sys

course_dir = Path.cwd() if (Path.cwd() / "runtime_example").is_dir() else Path.cwd() / "google-adk"
assert course_dir.is_dir(), "请从仓库根目录或 google-adk 目录运行 notebook"
if str(course_dir) not in sys.path:
    sys.path.insert(0, str(course_dir))
from importlib.metadata import version
assert version("google-adk") == "2.6.3"

## 1. 先理解应用中的三种数据

| 数据 | 保存位置 | 谁能改变 |
|---|---|---|
| 会话历史与事件 | `sessions.sqlite`，ADK DatabaseSessionService | Runner / SessionService |
| 订单、审批、退款 | `business.sqlite`，工具事务 | 经过业务校验的宿主/工具 |
| 政策、操作流程 | 第 13 章的资料源与第 14 章的 Skill 目录 | 经过审核的工程发布流程 |

Session 的历史证明“之前发生了哪些对话”；账本证明“业务已经执行了什么”。恢复历史之后仍必须查询账本，不能因为模型记得“已经退款”就跳过一致性检查。

```mermaid
flowchart LR
    H[HTTP 与身份认证] --> R[ADK Runner]
    R --> S[(Session 数据库)]
    R --> T[业务工具与授权]
    T --> B[(订单 / 审批 / 退款)]
    R --> K[政策检索 / Skill 指引]
```

代码入口：[server.py](capstone/server.py)、[agent.py](capstone/agent.py)、[store.py](capstone/store.py)。

## 2. 先跑通，再拆解

在仓库根目录运行：

```bash
cd google-adk
uv run --project .. python -m capstone.verify
```

这个命令默认就包含 HTTP 验收。它创建临时数据库、启动第一个服务进程，经真实 Runner 完成工具调用；停止服务后再启动第二个进程，读取之前的 Session，再发送同样请求。结束时回收自己启动的进程和临时文件。

预期检查：

1. 无凭据请求返回 401，其他租户看不到 Alice 的 Session。
2. 工具事件出现 Skill 加载、有效政策检索和退款，工具本体核对订单，退款账本写入金额 3000 分。
3. 两次服务的 PID 不同，第二个进程读到原来的历史事件；事件数量与内容指纹都一致，不只是数量相同。
4. 重试返回相同退款凭证，账本仍只有一行。
5. viewer、伪造审批和同 key 不同金额都被拒绝，账本不变。

这与依次运行若干独立练习不同：这些断言针对同一个应用、同一份持久数据。

In [ ]:
import tempfile
from capstone.verify import verify_business, verify_http

with tempfile.TemporaryDirectory(prefix="adk-capstone-") as directory:
    directory = Path(directory)
    verify_business(directory)
    verify_http(directory)


## 3. 模型替身究竟替代了什么

离线 `ScenarioModel` 接收固定 JSON 场景，依次生成函数调用和最后的结果。它替代的是模型网络调用；HTTP、身份绑定、ADK 工具协议、Session 和退款事务都实际执行。

因此离线通过证明工程接线与业务约束成立，**不证明模型能理解任意用户表达**。为控制费用，在线冒烟使用同一服务与业务存储的 `--business-only` 配置（查订单 → 退款），只把模型换成 LiteLlm；RAG/Skills 的真实模型轨迹在第 13、14 章单独验证：

```bash
# 工作目录 google-adk；至少两次用户请求，每次最多 4 次模型调用
uv run --project .. --env-file ../.env python -m capstone.verify --backend deepseek
```

在线失败要查看工具轨迹和账本，区分“没有调用工具”“参数不正确”“业务拒绝”“结果未正确表达”；不要为通过断言放宽权限。

## 4. 接入知识与 Skill 时，权限从哪里来

RAG 提供带来源的政策事实；Skill 提供按需加载的处理流程。两者都不能批准退款。政策文本中的“可以退”仍需要订单事实、可信身份、具体审批和余额检查。

第 13、14 章的独立验收保留，因为它们定位检索质量与渐进加载问题更直接。默认 HTTP 验收把两项能力连接到同一个售后 Agent、同一份 Session 和业务数据库；重启后的重复请求也经过 Skill/检索工具。viewer 虽能读取政策和加载流程，仍不能写入账本。运行入口和实际轨迹见 [Capstone README](capstone/README.md)。

**不要做的推导：** “检索命中了退款政策 → 用户有权限退款”；“模型加载了管理员 Skill → 当前身份变成管理员”。

## 5. 评估分三层，不互相替代

| 层次 | 入口 | 回答什么 |
|---|---|---|
| 工程验收 | `capstone.verify` | 同一业务应用是否恢复、授权、幂等正确 |
| ADK 官方评估 | [official_eval_example](official_eval_example/README.md) | 工具轨迹与回答是否达到配置阈值 |
| 售后数据集回归 | [eval_example](eval_example/README.md) | 多类任务/安全场景是否成功，业务账本是否匹配 |

冒烟集通过只说明所选案例通过；安全类没有样本时应显示“未评估”，不能显示 100%。完整发布门禁需要完整覆盖、阈值和相对基线的退化检查。固定模型的官方 eval 适合学习命令和制造稳定失败；真实模型质量要用在线数据集单独报告。

当前课程中的三个入口各有明确范围，不能把它们加总成一个没有定义的“生产就绪分”。

## 6. 把实验改成你能维护的工程

按顺序完成下面三个增量，每次只增加一种变化：

1. **新增“已过七天”的退款案例。** 先写预期：业务拒绝、账本不变；再补工具规则与回归数据。
2. **新增政策版本。** 让旧版本失效；比较固定检索集的 Recall@k 与引用合法性，再检查 Agent 是否仍依据有效资料回答。
3. **增加一个必须审批的 Skill 流程。** Skill 可以提示检查审批，但禁止把审批签发工具注册给模型。用恶意文档、伪造审批和普通 viewer 做联合反例。

每次提交都交付：改了哪个行为、对应失败案例、修复后的事件/账本证据，以及在线调用数。不要只交付一张“程序跑完”的截图。

## 评分矩阵

| 能力 | 尚未掌握 | 达到课程目标 | 进一步提升 |
|---|---|---|---|
| 解释机制 | 只能复述 API | 根据事件解释工具、状态和回调 | 能预测改动后的失败路径 |
| 数据边界 | 把对话当数据库 | 区分 Session、检索、业务账本 | 有数据版本与保留策略 |
| 验收 | 只看最终回答 | 同时断言轨迹、结果、副作用 | 有完整在线回归和退化门禁 |
| 故障恢复 | 同进程重跑 | 不同 PID、持久历史、幂等重试 | 支付渠道幂等与对账 |
| 安全 | 信任模型传入角色 | 可信身份、精确审批、工具复核 | 实际 IdP、审计与权限撤销 |

## 用五句话复述

重启后什么数据还在？重复请求为什么不重复退款？离线通过没有证明什么？RAG 与 Skill 各带来什么？什么证据才能支持发布决策？

完成本章后，回到 [00-框架全景与选型对比](../00-框架全景与选型对比.ipynb)，用你实际遇到的状态、恢复、评估和工具边界重新判断框架选择。